In [2]:
!pip install spacy
!python -m spacy download ru_core_news_sm 

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-1.0.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached confection-1.3.3-py3-none-any.whl.metadata (19 kB)
  Using cached cloudpathlib-0.24.0-py3-none-any.whl.metadata (16 kB)
  Using cached smart_open-7.6.1-py3-none-any.whl.metadata (25 kB)
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   --- ------------------------------------ 1.3/14.2 MB 11.8 MB/s eta 0:00:02
   ----------------------- ---------------- 8.4/14.2 MB 30.2 MB/s eta 0:00:01
   -------------------------------- ------- 11.5/14.2 MB 21.5 MB/s eta 0:00:01
   ---------------------------------------- 14.2/14.2 MB 20.6 MB/s eta 0:00:00
Using cached catalogue-2.0.10-py3-none-any.whl (17 kB)
Using cached confection-1.


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
     -- ------------------------------------- 1.0/15.3 MB 10.7 MB/s eta 0:00:02
     ------------------- -------------------- 7.3/15.3 MB 25.5 MB/s eta 0:00:01
     ----------------------- ---------------- 8.9/15.3 MB 18.0 MB/s eta 0:00:01
     --------------------------------------  15.2/15.3 MB 21.4 MB/s eta 0:00:01
     --------------------------------------- 15.3/15.3 MB 19.9 MB/s eta 0:00:00
  Using cached pymorphy3-2.0.6-py3-none-any.whl.metadata (2.4 kB)
  Using cached dawg2_python-0.9.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached pymorphy3_dicts_ru-2.4.417150.4580142-py2.py3-none-any.whl.metadata (2.0 kB)
Using cached pymorphy3-2.0.6-py3-none-any.whl (53 kB)
Using cached dawg2_python-0.9.0-py3-none-any.whl (9.3 kB)
Using cached pymorphy3_dicts_ru-2.4.417150.4580142-py2.py3-none-any.whl (8.4 MB)
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Предподготовка

Создание датасета: причинные связи однозначно определить только при наличии соответственных слов маркеров
Есть соответствующий файл с предложениями содержащими такие слова
Предложения обрабатываются, выделяются связи: выделяются грамматические основы частей предложений, между которыми стоит слово-маркер через дерево связей? выделяются временные связи по наличию в словаре временных меток
слова маркеры удаляются, но при этом связи сохраняются
СЦЕНАРИИ
Предложения со связями:
 - удаление союзов, разделение ссп на два простых предложения, изменение порядка предложений
 - удаление союзов, остальное не изменяется
 - тексты не меняются
 - разделение ссп на два простых предложения, при этом в часть с предлогом добавляется один из вариантов: "это произошло", "это случилось".
Предложение без связи около 15% данных:
 - предложения не меняются

Один файл с набором предложений input разбивается на множество файлов и хранятся в папке dataset/input_text

In [1]:
import os

input_file = "text.txt"
output_dir = "dataset/input_text"
os.makedirs(output_dir, exist_ok=True)

prefix = "text_"      
suffix = ".txt"       
padding = 5          

idx = 1  
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            filename = f"{prefix}{idx:0{padding}d}{suffix}"
            filepath = os.path.join(output_dir, filename)
            with open(filepath, "w", encoding="utf-8") as out:
                out.write(line)
            idx += 1 

ОБЩЕЕ ОПИСАНИЕ OUTPUT
Выжимка событий(вершины)
грамматическая основа + время(наличие в TIME_MARKERS, соответствие regex) + дополнение от глагола
Выжимка связей(ребра)
Если в предложении есть CAUSES_RELATIONSHIP - тип связи CAUSES, если в предложении есть TIME_RELATIONSHIP - тип связи TEMPORAL

РАЗБОР ПРЕДЛОЖЕНИЕ:
грамматическая основа:
подлежащее: nsubj, nsubj:pass, csubj, expl
сказуемое: ROOT, acl, ccomp, xcomp, advcl, csubj

дополнение при глаголе:
obj if token.head.pos == сказаемое

временная метка:
наличие в списке маркеров или соответствие regex

союз:
наличие в списке

ПРИМЕР РЕЗУЛЬТАТА
VERTEX_LIST:
V1: [временная_метка] [сказуемое] [подлежащее] [дополнение_при_глаголе]
V2: [временная_метка] [сказуемое] [подлежащее] [дополнение_при_глаголе]

RELATIONSHIPS_LIST:
V1->V2 [тип_связи]



**Пример извлечения отношений слов в предложении**

In [17]:
import spacy

nlp = spacy.load("ru_core_news_sm")

texts = ["Анатолий Писькин получил повышение в понедельник, ибо успешно завершил трудный проект. С тех пор как его назначили начальником, он стал меньше спать.", 
        "Завод снизил выбросы, так как фильтры обновили.",
        "Мои наушники не перестали работать, в связи с тем что перебился провод возле штекера. Пришлось купить новые, ибо ремонт обошёлся бы почти так же дорого."]

for text in texts:
    doc = nlp(text)

    print(f"Разбор предложения: {text}")
    print("-"*30)
    print(f"{'Слово':<12} | {'POS':<6} | {'Отношение (dep)':<18} | {'Главное слово':<15}")
    print("-"*30)

    relations = []
    for token in doc:
        if token.pos_ == "PUNCT":
            continue

        head_text = token.head.text if token.head != token else "ROOT"
        print(f"{token.text:<12} | {token.pos_:<6} | {token.dep_:<18} | {head_text:<15}")
        
        if token.head != token:
            relations.append((token.text, token.dep_, token.head.text))

    print("="*75)


Разбор предложения: Анатолий Писькин получил повышение в понедельник, ибо успешно завершил трудный проект. С тех пор как его назначили начальником, он стал меньше спать.
------------------------------
Слово        | POS    | Отношение (dep)    | Главное слово  
------------------------------
Анатолий     | PROPN  | nsubj              | получил        
Писькин      | PROPN  | appos              | Анатолий       
получил      | VERB   | ROOT               | ROOT           
повышение    | NOUN   | obj                | получил        
в            | ADP    | case               | понедельник    
понедельник  | NOUN   | obl                | получил        
ибо          | SCONJ  | mark               | завершил       
успешно      | ADV    | advmod             | завершил       
завершил     | VERB   | conj               | получил        
трудный      | ADJ    | amod               | проект         
проект       | NOUN   | obj                | завершил       
С            | ADV    | advmod      

**Словари маркеров**
типы связей:
 - временные связи(Temporal): BEFORE(до), AFTER(после), DURING(во время)

 - каузальные связи(Causal): CAUSES(вызывает), ENABLES(позволяет/способствует)

In [2]:
CAUSES = {
    "потому что", "так как", "поскольку", "ибо", "оттого что",
    "вследствие того что", "ввиду того что", "из-за того что", "по причине того что",
    "в связи с тем что", "в результате того что",
    "вследствие", "из-за", "ввиду", "по причине",
    "так что", "поэтому", "следовательно", "потому"
}

ENABLES = {
    "благодаря", "благодаря тому что",
    "за счет того что", "за счет",
    "посредством того что", "посредством",
    "путем того что", "путем",
    "при условии что", "если",
    "чтобы", "для того чтобы", "с тем чтобы", "дабы",
    "в силу того что", "на основании того что"
}

TEMPS  = {
    "пока", "лишь", "только", "едва", "как только", 
    "прежде чем", "в то время как", 
    "с тех пор как", "до тех пор пока", "по мере того как",
    "после того как", "когда"
    }

FLASHBACK = {"вспоминая", "оглядываясь назад", "задолго до",
             "несколько лет назад", "до этого", "ранее", "ещё в", 
             "вернёмся к тому что", "история началась с", "всё началось с того что", 
             "как вспоминает", "по словам", "накануне", "за год до этого", "за месяц до того", 
             "за неделю до этого", "до того как", "перед тем как", "в те времена", "в прошлом", "ранее в",
            "вернёмся к", "вернёмся к тому что",
            "история началась с", "история началась с того, что",
            "всё началось с того, что" "всё началось с того что",
            "как вспоминает", "вспоминая",
            "по словам", "по словам",
            "ещё в", "ещё в те времена",
            "задолго до", "задолго до этого",
            "ранее в", "ранее в том же году"}

In [3]:
import re
import spacy

ALL_CONJS = sorted(set(CAUSES) | set(TEMPS) | set(ENABLES) | set(FLASHBACK), key=len, reverse=True)
CONJ_RE = re.compile(r"\b(" + "|".join(map(re.escape, ALL_CONJS)) + r")\b", re.IGNORECASE)

TIME_RE = re.compile(r"""(?ix)
    (?:до|к|с|около|после|перед|в|на)\s+\d{1,2}\s*(?:утра?|дня?|вечера?|ночи?|полдня?|полночи?)
    |
    (?:\d{1,4}\s*|пару|несколько|пол)?\s*(?:год[ау]?|г\.|месяц[аы]?|недел[ьюи]?|дн[еяй]?|час[аов]?|минут[аы]?|секунд[аы]?|век[ау]?|столетие[м]?|полгода?)\s+(?:назад|тому)
    |
    (?:в|на|после|до|к|со|с|у|около|перед|через|спустя)\s+\d{1,4}\s*
    (?:год[ау]?|г\.|века?|столетие[м]?|час[аов]?|минут[аы]?|секунд[аы]?|дн[ея]?|недел[ья]?|месяц[еаы]?|квартал[еах]?)?
    |
    (?:в|на)\s+(?:прошлом|прошедшем|будущем|следующем|нынешнем|текущем)\s+(?:году|месяце|неделе|веке|квартале|полугоди)
    |
    (?:в|на|до|после|к|с|у|около|перед|через)\s+(?:январ[еяю]|феврал[еяю]|март[еау]?|апрел[еяю]|ма[ею]|июн[еяю]|июл[еяю]|август[еау]?|сентябр[еяю]|октябр[еяю]|ноябр[еяю]|декабр[еяю])
    |
    \d{1,2}:\d{2}
    |
    (?:сегодня|завтра|вчера|позавчера|послезавтра|утром|вечером|ночью|днем|сейчас|раньше|позже|на днях|в прошлом|в будущем|недел[ияю]|месяц[еаы]?)
""")
VERB_RE = re.compile(r"\b[а-яёА-ЯЁ]*[а-яёА-ЯЁ]*(?:л|л[а-яё]|т|ет|ют|ят|шь|ет|ут|ют|ем|ём|ете|ёте|ли|ли[а-яё]|сь|лся|лась|лись|но|нно|то|тый|тая|тое|тые)\b")

nlp = spacy.load("ru_core_news_sm")

def get_rel_type(conj_word):
    cw = conj_word.lower()
    if cw in {c.lower() for c in FLASHBACK}: return "flashback"
    if cw in {c.lower() for c in CAUSES}:    return "causes"
    if cw in {c.lower() for c in TEMPS}:     return "temporal"
    if cw in {c.lower() for c in ENABLES}:   return "enables"
    return None

def normalize_time(raw):
    if not raw: return "-"
    return re.sub(r'\s+', ' ', raw.strip().lower())

def has_verb(text):
    if not text or len(text) < 3: return False
    if VERB_RE.search(text): return True  
    return any(t.pos_ in ("VERB", "AUX") for t in nlp(text))

def normalize_person(raw):
    if not raw: return "-"
    doc = nlp(raw)
    for token in doc:
        if token.pos_ == "PROPN" and token.dep_ in ("nsubj", "nsubj:pass"):
            name_parts = [token.text]
            idx = token.i + 1
            while idx < len(doc) and doc[idx].pos_ == "PROPN":
                if doc[idx].head.i == token.i or doc[idx].dep_ in ("flat:name", "compound", "appos"):
                    name_parts.append(doc[idx].text)
                    idx += 1
                else:
                    break
            return " ".join(name_parts)
    return "-"

def extract(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    vertices = []
    relations = []
    vid_counter = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence: continue

        raw_clauses = re.split(r'[,;:]\s*', sentence)
        clauses = [c.strip() for c in raw_clauses if c.strip()]

        verb_clauses = [(i, c) for i, c in enumerate(clauses) if has_verb(c)]

        for rank, (orig_idx, clause) in enumerate(verb_clauses):
            vid_counter += 1
            curr_id = vid_counter

            conj_match = CONJ_RE.match(clause)
            conj_type = None
            clean_clause = clause

            if conj_match:
                conj_word = conj_match.group(0)
                conj_type = get_rel_type(conj_word)
                clean_clause = clause[conj_match.end():].strip()
                clean_clause = re.sub(r'^[,\s]+', '', clean_clause)

            if not clean_clause.strip():
                vid_counter -= 1
                continue

            tm = TIME_RE.search(clean_clause)
            time_str = tm.group(0) if tm else ""
            text_only = clean_clause.replace(time_str, "", 1).strip() if time_str else clean_clause

            person = normalize_person(text_only)
            text_final = re.sub(r'[.,;:!?]+', ' ', text_only).strip().lower()
            text_final = re.sub(r'\s+', ' ', text_final)

            vertices.append({
                "id": curr_id,
                "time": normalize_time(time_str),
                "person": person,
                "text": text_final
            })

            if conj_type:
                if conj_type == "flashback":
                    if curr_id > 1:
                        relations.append((curr_id, curr_id - 1, "flashback"))
                elif rank == 0:
                    relations.append((curr_id, curr_id + 1, conj_type))
                else:
                    relations.append((curr_id, curr_id - 1, conj_type))

        relations[:] = [r for r in relations if r[1] <= vid_counter]

    out = ["vertex_list:"]
    for v in vertices:
        out.append(f"v{v['id']}: {v['time']} | {v['person']} | {v['text']}")

    out.append("\nrelationship_list:")
    if relations:
        for v1, v2, r in relations:
            out.append(f"v{v1}->v{v2} {r}")
    else:
        out.append("(нет связей)")

    return "\n".join(out)

if __name__ == "__main__":
    tests = [
        "Сергей Иванов купил машину, несколько лет назад Алексей Кузнецова готовила ужин. Николай козлов путешествовал, вернёмся к тому что анна павлова смотрела фильм.",
        "В 1973 году ольга Волков выехала на встречную полосу, по причине Анна Смирнов проехала на красный свет, по причине Ксения Лебедев признала вину, вследствие Алексей Фёдоров заплатил половину штрафа.",
        "В 2023 году компания вышла на рынок. За три года до этого она была лишь стартапом в гараже.",
        "Суд вынес решение вчера. Ранее в прошлом году свидетели давали противоречивые показания, потому что ключевые документы были утеряны."
    ]

    for i, txt in enumerate(tests, 1):
        print(f"Тест {i}: {txt}")
        print(extract(txt))
        print("=" * 80)

🧪 Тест 1: Сергей Иванов купил машину, несколько лет назад Алексей Кузнецова готовила ужин. Николай козлов путешествовал, вернёмся к тому что анна павлова смотрела фильм.
vertex_list:
v1: - | Сергей Иванов | сергей иванов купил машину
v2: - | Алексей Кузнецова | алексей кузнецова готовила ужин
v3: - | Николай козлов | николай козлов путешествовал
v4: - | анна павлова | анна павлова смотрела фильм

relationship_list:
v2->v1 flashback
v4->v3 flashback
🧪 Тест 2: В 1973 году ольга Волков выехала на встречную полосу, по причине Анна Смирнов проехала на красный свет, по причине Ксения Лебедев признала вину, вследствие Алексей Фёдоров заплатил половину штрафа.
vertex_list:
v1: в 1973 году | ольга Волков | ольга волков выехала на встречную полосу
v2: - | Анна Смирнов | анна смирнов проехала на красный свет
v3: - | Ксения Лебедев | ксения лебедев признала вину
v4: - | Алексей Фёдоров | алексей фёдоров заплатил половину штрафа

relationship_list:
v2->v1 causes
v3->v2 causes
v4->v3 causes
🧪 Тест 3

ВЫДЕЛЕНИЕ ВСЕХ СВЯЗЕЙ И ФАЙЛОВ В OUTPUT_TEXT

In [4]:
import glob

input_dir = "dataset/input_text"
output_dir = "dataset/output_text"
os.makedirs(output_dir, exist_ok=True)

input_files = sorted(glob.glob(os.path.join(input_dir, "text_*.txt")))

if not input_files:
    print(f"В папке {input_dir} не найдено файлов text_*.txt")
else:
    print(f"Найдено файлов: {len(input_files)}")
    for in_path in input_files:
        basename = os.path.basename(in_path)      
        name, ext = os.path.splitext(basename)    
        out_name = f"graph_{name}{ext}"              
        out_path = os.path.join(output_dir, out_name)

        with open(in_path, "r", encoding="utf-8") as f:
            text = f.read().strip()

        if text:
            result = extract(text)
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(result)
            print(f"{basename} -> {out_name}")
        else:
            print(f"{basename} пустой, пропущен")

Найдено файлов: 21821
text_00001.txt -> graph_text_00001.txt
text_00002.txt -> graph_text_00002.txt
text_00003.txt -> graph_text_00003.txt
text_00004.txt -> graph_text_00004.txt
text_00005.txt -> graph_text_00005.txt
text_00006.txt -> graph_text_00006.txt
text_00007.txt -> graph_text_00007.txt
text_00008.txt -> graph_text_00008.txt
text_00009.txt -> graph_text_00009.txt
text_00010.txt -> graph_text_00010.txt
text_00011.txt -> graph_text_00011.txt
text_00012.txt -> graph_text_00012.txt
text_00013.txt -> graph_text_00013.txt
text_00014.txt -> graph_text_00014.txt
text_00015.txt -> graph_text_00015.txt
text_00016.txt -> graph_text_00016.txt
text_00017.txt -> graph_text_00017.txt
text_00018.txt -> graph_text_00018.txt
text_00019.txt -> graph_text_00019.txt
text_00020.txt -> graph_text_00020.txt
text_00021.txt -> graph_text_00021.txt
text_00022.txt -> graph_text_00022.txt
text_00023.txt -> graph_text_00023.txt
text_00024.txt -> graph_text_00024.txt
text_00025.txt -> graph_text_00025.txt
tex

Датасет готов: входные тексты dataset/input_text -> события и связи dataset/output_text
Однако стоит отметить, что по большей части в обычных текстах где есть какая либо связь не используются слова маркеры, поэтому для половины текстов датасета мы удалим слова маркеры, при этом связи останутся прежними, так как слова маркеры не фигурируют в итоговом графе, так же стоит отметить, что при удалении слов маркеров связь не пропадает.

In [5]:
import os
import re
import glob
import shutil

markers = sorted(list(CAUSES | TEMPS | ENABLES), key=len, reverse=True)
MARKER_PATTERN = re.compile(
    r'(?<!\w)(?:' + '|'.join(map(re.escape, markers)) + r')(?!\w)', 
    re.IGNORECASE
)

def clean_text(text):
    cleaned = MARKER_PATTERN.sub('', text)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = re.sub(r'\s*([.,;:!?»"\)])\s*', r'\1 ', cleaned)
    cleaned = re.sub(r'\s*([«\(])\s*', r'\1', cleaned)
    return cleaned.strip()

def extract_file_number(filename):
    match = re.search(r'text_(\d+)\.txt', filename)
    return int(match.group(1)) if match else float('inf')

input_dir = "dataset/input_text"
output_dir = "dataset/input_text_clean"
os.makedirs(output_dir, exist_ok=True)

all_files = sorted(glob.glob(os.path.join(input_dir, "text_*.txt")), 
                   key=lambda p: extract_file_number(os.path.basename(p)))

if not all_files:
    print(f"В папке {input_dir} не найдено файлов text_*.txt")
else:
    print(f"Найдено файлов: {len(all_files)}")
    print(f"Диапазоны для очистки: 1–6000 и 12000–18000")
    print(f"Остальные файлы: копируются без изменений")
    print("-" * 50)
    
    processed = skipped = 0
    
    for filepath in all_files:
        filename = os.path.basename(filepath)
        file_num = extract_file_number(filename)
        out_path = os.path.join(output_dir, filename)
        
        try:
            if (1 <= file_num <= 6000) or (12000 <= file_num <= 18000):
                with open(filepath, "r", encoding="utf-8") as f:
                    text = f.read()
                cleaned = clean_text(text)
                with open(out_path, "w", encoding="utf-8") as f:
                    f.write(cleaned)
                processed += 1
            else:
                shutil.copy2(filepath, out_path)
                skipped += 1
                
        except Exception as e:
            print(f"Ошибка в файле {filename}: {e}")

    print(f"Обработано (удалены маркеры): {processed} файлов")
    print(f"Скопировано (без изменений): {skipped} файлов")
    print(f"Результат в папке: {output_dir}")

📂 Найдено файлов: 21821
🔹 Диапазоны для очистки: 1–5000 и 11000–18000
🔹 Остальные файлы: копируются без изменений
--------------------------------------------------

✅ Готово!
   • Обработано (удалены маркеры): 12001 файлов
   • Скопировано (без изменений): 9820 файлов
   • Результат в папке: dataset/input_text_clean


In [6]:
import json
from pathlib import Path

MAX_CHARS = 5000

INPUT_DIR = Path("dataset/input_text_clean")
OUTPUT_DIR = Path("dataset/output_text")
OUT_JSONL = Path("df_text2graph.jsonl")

jsonl_lines = []
skipped = []

input_files = sorted(INPUT_DIR.glob("text_*.txt"))
print(f"Найдено входных файлов: {len(input_files)}")

for input_file in input_files:
    case_id = input_file.stem     
    number = case_id.replace("text_", "")
    output_file = OUTPUT_DIR / f"graph_text_{number}.txt"

    if not output_file.exists():
        skipped.append(f"{case_id}: output not found")
        continue

    try:
        input_text = input_file.read_text(encoding="utf-8", errors="ignore").strip()
        input_text = input_text[:MAX_CHARS] 

        output_text = output_file.read_text(encoding="utf-8", errors="ignore").strip()
        if not output_text:
            skipped.append(f"{case_id}: empty output")
            continue

        pair = {
            "id": case_id,
            "input": input_text,
            "output": output_text
        }
        jsonl_lines.append(json.dumps(pair, ensure_ascii=False))

    except Exception as e:
        skipped.append(f"{case_id}: {str(e)}")

OUT_JSONL.write_text("\n".join(jsonl_lines), encoding="utf-8")
print(f"Сохранено пар: {len(jsonl_lines)}")
print(f"Пропущено:     {len(skipped)}")
print(f"Результат:    {OUT_JSONL}")

Найдено входных файлов: 21821
Сохранено пар: 21821
Пропущено:     0
Результат:    df_text2graph.jsonl


In [121]:
import random

# Исходный список FLASHBACK (все элементы из задания)
FLASHBACK_RAW = {
    "вспоминая", "оглядываясь назад", "задолго до",
    "несколько лет назад", "до этого", "ранее", "ещё в",
    "вернёмся к", "история началась с", "всё началось с того что",
    "как вспоминает", "по словам", "накануне", "за год до этого",
    "за месяц до того", "за неделю до этого", "до того как",
    "перед тем как", "в те времена", "в прошлом", "ранее в"
}

# Преобразуем в список для случайного выбора
FLASHBACK_LIST = list(FLASHBACK_RAW)

# Словарь правил для сложных связок
# Ключ — исходная фраза, значение — шаблон, где {0} — первая основа, {1} — вторая основа
FLASHBACK_TEMPLATES = {
    "вернёмся к": "{0}, вернёмся к тому что {1}",
    "история началась с": "{0}, история началась с того, что {1}",
    "всё началось с того что": "{0}, всё началось с того что {1}",
    "как вспоминает": "{0}, вспоминая {1}",
    "по словам": "{0}, по словам {1}",
    "ещё в": "{0}, ещё в те времена {1}",
    "задолго до": "{0}, задолго до этого {1}",
    "ранее в": "{0}, ранее в том же году {1}",
}

# Для остальных связок используем простой шаблон: основа1, связка основа2
DEFAULT_TEMPLATE = "{0}, {conn} {1}"

# Имена и фамилии
FIRST_NAMES = ["Иван", "Елена", "Пётр", "Анна", "Дмитрий", "Ольга", "Сергей", "Татьяна",
               "Алексей", "Мария", "Николай", "Ксения", "Владимир", "Наталья", "Михаил", "Евгения"]
LAST_NAMES = ["Иванов", "Петрова", "Сидоров", "Кузнецова", "Смирнов", "Васильева",
              "Попов", "Новикова", "Фёдоров", "Морозова", "Волков", "Алексеева",
              "Лебедев", "Егорова", "Козлов", "Павлова"]

# Глаголы (прошедшее время, мужской/женский род)
VERBS_MASC = [
    "работал", "отдыхал", "учился", "путешествовал", "читал книгу",
    "готовил ужин", "смотрел фильм", "играл на гитаре", "писал письмо",
    "встретил друга", "купил машину", "построил дом", "защитил диссертацию"
]
VERBS_FEM = [
    "работала", "отдыхала", "училась", "путешествовала", "читала книгу",
    "готовила ужин", "смотрела фильм", "играла на гитаре", "писала письмо",
    "встретила подругу", "купила машину", "построила дом", "защитила диссертацию"
]

def get_verb(first_name, last_name):
    """Возвращает глагол в правильном роде."""
    if last_name.endswith(('а', 'я')) or first_name in ["Елена", "Анна", "Ольга", "Татьяна", "Мария", "Ксения", "Наталья", "Евгения"]:
        return random.choice(VERBS_FEM)
    else:
        return random.choice(VERBS_MASC)

def generate_clause():
    """Генерирует одну грамматическую основу: имя + фамилия + глагол."""
    first = random.choice(FIRST_NAMES)
    last = random.choice(LAST_NAMES)
    verb = get_verb(first, last)
    return f"{first} {last} {verb}"

def make_sentence_with_flashback(clause1, clause2, connector):
    """Соединяет две основы с помощью flashback-связки."""
    if connector in FLASHBACK_TEMPLATES:
        template = FLASHBACK_TEMPLATES[connector]
        return template.format(clause1, clause2).capitalize() + "."
    else:
        # Для простых связок (наречия, деепричастия)
        result = f"{clause1}, {connector} {clause2}"
        # Добавляем точку и заглавную букву
        return result[0].upper() + result[1:] + "."

def generate_sentence():
    """Генерирует одно предложение с двумя основами и случайной flashback-связкой."""
    c1 = generate_clause()
    c2 = generate_clause()
    # Убедимся, что второе подлежащее не совпадает с первым (необязательно, но для разнообразия)
    while c1.split()[0] == c2.split()[0]:  # сравниваем первые слова (имя)
        c2 = generate_clause()
    connector = random.choice(FLASHBACK_LIST)
    return make_sentence_with_flashback(c1, c2, connector)

# Генерация 1000 текстов (каждый текст — два предложения)
texts = []
for _ in range(1000):
    sent1 = generate_sentence()
    sent2 = generate_sentence()
    texts.append(f"{sent1} {sent2}")


print("\n".join(texts))

Мария Новикова училась, в прошлом Владимир Смирнов готовил ужин. Пётр Новикова встретила подругу, перед тем как Елена Иванов защитила диссертацию.
Пётр козлов встретил друга, ещё в те времена мария смирнов защитила диссертацию. Евгения Фёдоров работала, за месяц до того Пётр Павлова читала книгу.
Николай Фёдоров читал книгу, ранее Иван Иванов читал книгу. Иван Морозова отдыхала, до этого Наталья Смирнов путешествовала.
Мария Кузнецова работала, до этого Владимир Фёдоров купил машину. Николай Васильева училась, вспоминая Евгения Петрова построила дом.
Алексей Сидоров работал, до этого Елена Петрова путешествовала. Сергей фёдоров встретил друга, задолго до этого елена сидоров писала письмо.
Николай Новикова отдыхала, за год до этого Пётр Иванов смотрел фильм. Пётр Павлова готовила ужин, несколько лет назад Сергей Иванов смотрел фильм.
Пётр Васильева играла на гитаре, за год до этого Ксения Егорова купила машину. Наталья Петрова отдыхала, оглядываясь назад Михаил Кузнецова купила машину.
